In [1]:
!pip install streamlit sqlalchemy pandas scikit-learn numpy joblib -q
print("✅ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 41.6 MB/s eta 0:00:00
✅ Done


In [2]:
code = """
from sqlalchemy import create_engine, Column, Integer, String, Float, Date, Text
from sqlalchemy.orm import declarative_base, sessionmaker

DATABASE_URL = "sqlite:///./mira_health.db"
engine       = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base         = declarative_base()

class Patient(Base):
    __tablename__ = "patients"
    id            = Column(Integer, primary_key=True, index=True)
    full_name     = Column(String(100), nullable=False)
    date_of_birth = Column(Date, nullable=False)
    email         = Column(String(150), unique=True, nullable=False)
    age           = Column(Integer, nullable=False)
    glucose       = Column(Float, nullable=False)
    haemoglobin   = Column(Float, nullable=False)
    cholesterol   = Column(Float, nullable=False)
    remarks       = Column(Text, nullable=True)

def init_db():
    Base.metadata.create_all(bind=engine)

def get_session():
    return SessionLocal()

def create_patient(db, full_name, dob, email, age, glucose, haemoglobin, cholesterol, remarks):
    p = Patient(full_name=full_name, date_of_birth=dob, email=email,
                age=age, glucose=glucose, haemoglobin=haemoglobin,
                cholesterol=cholesterol, remarks=remarks)
    db.add(p); db.commit(); db.refresh(p)
    return p

def get_all_patients(db):
    return db.query(Patient).all()

def get_patient_by_id(db, pid):
    return db.query(Patient).filter(Patient.id == pid).first()

def get_patient_by_email(db, email):
    return db.query(Patient).filter(Patient.email == email).first()

def update_patient(db, pid, **kwargs):
    p = get_patient_by_id(db, pid)
    if not p: return None
    for k, v in kwargs.items():
        setattr(p, k, v)
    db.commit(); db.refresh(p)
    return p

def delete_patient(db, pid):
    p = get_patient_by_id(db, pid)
    if not p: return False
    db.delete(p); db.commit()
    return True
"""
with open("database.py", "w") as f:
    f.write(code)
print("✅ database.py written")

✅ database.py written


In [3]:
code = """
import re
from datetime import date
from typing import Tuple

EMAIL_REGEX = re.compile(r"^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$")

def validate_email(email):
    if not email or not EMAIL_REGEX.match(email.strip()):
        return False, "Please enter a valid email address."
    return True, ""

def validate_dob(dob):
    if dob is None:
        return False, "Date of birth is required."
    if dob >= date.today():
        return False, "Date of birth cannot be today or a future date."
    if (date.today() - dob).days // 365 > 120:
        return False, "Please enter a realistic date of birth."
    return True, ""

def validate_numeric(value, field_name, min_val, max_val):
    try:
        v = float(value)
    except (TypeError, ValueError):
        return False, f"{field_name} must be a numeric value."
    if v < min_val or v > max_val:
        return False, f"{field_name} must be between {min_val} and {max_val}."
    return True, ""

def validate_patient_inputs(full_name, dob, email, glucose, haemoglobin, cholesterol):
    errors = []
    if not full_name or not full_name.strip():
        errors.append("Full name is required.")
    ok, msg = validate_dob(dob);                              errors += [msg] if not ok else []
    ok, msg = validate_email(email);                          errors += [msg] if not ok else []
    ok, msg = validate_numeric(glucose,     "Glucose",     50,  500); errors += [msg] if not ok else []
    ok, msg = validate_numeric(haemoglobin, "HbA1c",       3.0, 15.0); errors += [msg] if not ok else []
    ok, msg = validate_numeric(cholesterol, "Cholesterol", 100, 400);  errors += [msg] if not ok else []
    return len(errors) == 0, errors
"""
with open("validators.py", "w") as f:
    f.write(code)
print("✅ validators.py written")

✅ validators.py written


In [4]:
code = """
import os, numpy as np, pandas as pd, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

MODEL_PATH  = "model/rf_health_model.pkl"
SCALER_PATH = "model/scaler.pkl"
np.random.seed(42)

def generate_synthetic_dataset(n=5000):
    print("Generating dataset based on WHO/ADA/Kaggle clinical distributions...")
    n0 = int(n * 0.60)
    n1 = int(n * 0.15)
    n2 = n - n0 - n1
    df = pd.DataFrame({
        "age":         np.concatenate([np.random.randint(20,65,n0), np.random.randint(35,70,n1), np.random.randint(40,80,n2)]),
        "glucose":     np.concatenate([np.random.normal(88,12,n0).clip(65,99), np.random.normal(112,8,n1).clip(100,125), np.random.normal(175,35,n2).clip(126,350)]),
        "haemoglobin": np.concatenate([np.random.normal(5.2,0.4,n0).clip(4.0,5.6), np.random.normal(5.9,0.3,n1).clip(5.7,6.4), np.random.normal(7.8,1.0,n2).clip(6.5,12)]),
        "cholesterol": np.concatenate([np.random.normal(185,20,n0).clip(150,199), np.random.normal(220,15,n1).clip(200,239), np.random.normal(255,25,n2).clip(240,320)]),
        "label":       np.concatenate([np.zeros(n0+n1,dtype=int), np.ones(n2,dtype=int)]),
    })
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

def train_and_save():
    os.makedirs("model", exist_ok=True)
    df = generate_synthetic_dataset(5000)
    X  = df[["age","glucose","haemoglobin","cholesterol"]].values
    y  = df["label"].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)
    clf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=5,
                                  random_state=42, class_weight="balanced")
    clf.fit(X_train_s, y_train)
    print(f"Accuracy: {accuracy_score(y_test, clf.predict(X_test_s)):.2%}")
    print(classification_report(y_test, clf.predict(X_test_s), target_names=["No Diabetes","Diabetic"]))
    joblib.dump(clf,    MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)
    print("Model saved!")
    return clf, scaler

if __name__ == "__main__":
    train_and_save()
"""
with open("train_model.py", "w") as f:
    f.write(code)
print("✅ train_model.py written")

✅ train_model.py written


In [24]:
import os, numpy as np, joblib
from datetime import date

MODEL_PATH  = "model/rf_health_model.pkl"
SCALER_PATH = "model/scaler.pkl"
_clf    = None
_scaler = None

def _load_model():
    global _clf, _scaler
    if _clf is None:
        if not os.path.exists(MODEL_PATH):
            import train_model
            train_model.train_and_save()
        _clf    = joblib.load(MODEL_PATH)
        _scaler = joblib.load(SCALER_PATH)

def _glucose_detail(g):
    if g < 70:     return ("Low — Hypoglycaemia",        "⚠️", "Below 70 mg/dL")
    elif g <= 99:  return ("Normal",                     "✅", "70–99 mg/dL")
    elif g <= 125: return ("Pre-Diabetic Range",         "⚠️", "100–125 mg/dL")
    else:          return ("Diabetic Range",             "🔴", "≥126 mg/dL")

def _hba1c_detail(h):
    if h < 5.7:    return ("Normal",                     "✅", "Below 5.7%")
    elif h <= 6.4: return ("Pre-Diabetic",               "⚠️", "5.7–6.4%")
    else:          return ("Diabetic",                   "🔴", "≥6.5%")

def _cholesterol_detail(c):
    if c < 200:    return ("Desirable",                  "✅", "Below 200 mg/dL")
    elif c <= 239: return ("Borderline High",            "⚠️", "200–239 mg/dL")
    else:          return ("High — Cardiovascular Risk", "🔴", "≥240 mg/dL")

def _age_context(age):
    if age < 30:   return "Low baseline metabolic risk. Routine annual screening advised."
    elif age < 45: return "Moderate risk window. Biennial screening recommended."
    elif age < 60: return "Elevated risk group. Annual screening essential."
    else:          return "High-risk age group. Regular check-ups every 6–12 months advised."

def predict_health(age, glucose, haemoglobin, cholesterol):
    _load_model()
    X        = np.array([[age, glucose, haemoglobin, cholesterol]], dtype=float)
    X_scaled = _scaler.transform(X)
    prediction = _clf.predict(X_scaled)[0]
    proba      = _clf.predict_proba(X_scaled)[0]
    risk_pct   = proba[1] * 100

    today = date.today().strftime("%d %B %Y")

    if prediction == 0 and risk_pct < 25:
        risk_level = "LOW RISK"
        risk_label = "No Diabetes Detected"
        risk_icon  = "✅"
        recommendation = (
            "All biomarkers are within acceptable clinical ranges.\n"
            "Maintain a balanced diet, regular physical activity (150 min/week),\n"
            "stay hydrated, and schedule an annual health check-up."
        )
    elif prediction == 0 and risk_pct < 50:
        risk_level = "MILD RISK"
        risk_label = "Borderline Values Detected"
        risk_icon  = "⚠️"
        recommendation = (
            "Some biomarkers are approaching threshold levels.\n"
            "Lifestyle modifications are advised — reduce refined carbohydrates,\n"
            "increase physical activity, and schedule a follow-up blood test in 3–6 months."
        )
    elif prediction == 1 and risk_pct < 75:
        risk_level = "MODERATE-HIGH RISK"
        risk_label = "Diabetes Indicators Present"
        risk_icon  = "🔶"
        recommendation = (
            "Multiple biomarkers indicate elevated diabetes risk.\n"
            "Consult a physician promptly for an HbA1c confirmation test,\n"
            "fasting plasma glucose test, and a full lipid panel review."
        )
    else:
        risk_level = "HIGH RISK"
        risk_label = "Strong Diabetes Indicators"
        risk_icon  = "🔴"
        recommendation = (
            "Critical biomarker levels detected across multiple parameters.\n"
            "Immediate medical consultation is strongly advised.\n"
            "Do not delay — early treatment significantly improves outcomes."
        )

    g_status, g_icon, g_range = _glucose_detail(glucose)
    h_status, h_icon, h_range = _hba1c_detail(haemoglobin)
    c_status, c_icon, c_range = _cholesterol_detail(cholesterol)

    report = (
        f"╔══════════════════════════════════════════════════════════╗\n"
        f"  {risk_icon}  ASSESSMENT RESULT: {risk_level}\n"
        f"     {risk_label}\n"
        f"╚══════════════════════════════════════════════════════════╝\n"
        f"\n"
        f"  ML Confidence    : {risk_pct:.1f}% diabetes risk\n"
        f"  Model            : Random Forest Classifier (100 trees)\n"
        f"  Report Date      : {today}\n"
        f"\n"
        f"──────────────────────────────────────────────────────────\n"
        f"  BIOMARKER ANALYSIS\n"
        f"──────────────────────────────────────────────────────────\n"
        f"  {g_icon} Glucose       : {glucose} mg/dL   →  {g_status} ({g_range})\n"
        f"  {h_icon} HbA1c         : {haemoglobin}%       →  {h_status} ({h_range})\n"
        f"  {c_icon} Cholesterol   : {cholesterol} mg/dL  →  {c_status} ({c_range})\n"
        f"  📅 Age           : {int(age)} yrs       →  {_age_context(int(age))}\n"
        f"\n"
        f"──────────────────────────────────────────────────────────\n"
        f"  CLINICAL RECOMMENDATION\n"
        f"──────────────────────────────────────────────────────────\n"
        f"  {recommendation}\n"
        f"\n"
        f"──────────────────────────────────────────────────────────\n"
        f"  ⚕️  DISCLAIMER\n"
        f"──────────────────────────────────────────────────────────\n"
        f"  This report is generated by an AI/ML system for\n"
        f"  informational purposes only. It does not substitute\n"
        f"  professional medical diagnosis or treatment.\n"
        f"  Always consult a qualified healthcare provider.\n"
    )
    return report

In [25]:
%%writefile app.py
import streamlit as st
import pandas as pd
from datetime import date

from database import init_db, get_session, create_patient, get_all_patients, \
    get_patient_by_id, get_patient_by_email, update_patient, delete_patient
from ml_engine import predict_health
from validators import validate_patient_inputs

st.set_page_config(page_title="Health Prediction", page_icon="🏥",
                   layout="wide", initial_sidebar_state="expanded")
init_db()

st.markdown("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

    html, body, [class*="css"] { font-family: 'Inter', sans-serif; }

    .stApp { background: #f0f4f8; }

    /* ── Sidebar ── */
    div[data-testid="stSidebarContent"] {
        background: linear-gradient(180deg, #0f2942 0%, #0d4f6e 100%);
        padding-top: 1rem;
    }
    div[data-testid="stSidebarContent"] * { color: #e8f4f8 !important; }
    div[data-testid="stSidebarContent"] hr { border-color: rgba(255,255,255,0.15) !important; }

    /* ── Header ── */
    .main-header {
        background: linear-gradient(135deg, #0f2942 0%, #0d6e6e 60%, #0a8f7a 100%);
        padding: 2rem 2.5rem; border-radius: 16px; margin-bottom: 1.5rem;
        box-shadow: 0 4px 20px rgba(15,41,66,0.25);
    }
    .main-header h1 { margin:0; font-size:1.9rem; font-weight:700; color:white !important; letter-spacing:-0.5px; }
    .main-header p  { margin:0.4rem 0 0; font-size:0.9rem; color:rgba(255,255,255,0.75) !important; }
    .header-badges  { display:flex; gap:0.6rem; flex-wrap:wrap; margin-top:1rem; }
    .badge {
        background:rgba(255,255,255,0.15); border:1px solid rgba(255,255,255,0.25);
        border-radius:20px; padding:0.25rem 0.75rem;
        font-size:0.75rem; color:white !important; font-weight:500;
    }

    /* ── Fix form labels ── */
    .stTextInput label, .stDateInput label,
    .stNumberInput label, .stSelectbox label,
    .stRadio label, .stCheckbox label {
        color: #0f2942 !important; font-weight:600 !important; font-size:0.85rem !important;
    }
    p, .stMarkdown p { color: #334155; }
    .stCaption, .stCaption p { color:#64748b !important; font-size:0.78rem !important; }

    /* ── Metrics ── */
    div[data-testid="metric-container"] {
        background:white; border:1px solid #e2e8f0; border-radius:12px;
        padding:1.2rem 1.5rem !important; box-shadow:0 1px 4px rgba(0,0,0,0.06);
    }
    [data-testid="stMetricLabel"] p {
        color:#64748b !important; font-size:0.75rem !important;
        font-weight:700 !important; text-transform:uppercase; letter-spacing:0.05em;
    }
    [data-testid="stMetricValue"] { color:#0f2942 !important; font-size:1.7rem !important; font-weight:700 !important; }

    /* ── Cards ── */
    .card {
        background:white; border-radius:14px; border:1px solid #e2e8f0;
        padding:1.5rem; box-shadow:0 1px 6px rgba(0,0,0,0.05); margin-bottom:1rem;
    }
    .card-title {
        font-size:0.7rem; font-weight:700; text-transform:uppercase;
        letter-spacing:0.08em; color:#64748b; margin-bottom:0.75rem;
    }

    /* ── Risk report box ── */
    .risk-box {
        background:#f8fafc;
        border:1px solid #cbd5e1;
        border-left:5px solid #64748b;
        border-radius:12px;
        padding:1.5rem 1.75rem;
        white-space:pre-wrap;
        font-family:'JetBrains Mono', monospace;
        font-size:0.78rem;
        color:#1e293b !important;
        line-height:1.85;
        margin-top:0.75rem;
        letter-spacing:0.01em;
    }
    .risk-low  { border-left-color:#10b981 !important; background:#f0fdf4 !important; border-color:#86efac !important; color:#052e16 !important; }
    .risk-mild { border-left-color:#f59e0b !important; background:#fffbeb !important; border-color:#fde68a !important; color:#451a03 !important; }
    .risk-high { border-left-color:#ef4444 !important; background:#fef2f2 !important; border-color:#fca5a5 !important; color:#450a0a !important; }

    /* ── Biomarker pills ── */
    .biomarker-row { display:flex; gap:0.75rem; flex-wrap:wrap; margin:0.75rem 0; }
    .bio-pill {
        border-radius:10px; padding:0.65rem 1.1rem;
        display:flex; flex-direction:column; align-items:center; min-width:130px;
        box-shadow:0 1px 4px rgba(0,0,0,0.06);
    }
    .bio-pill .blabel  { font-size:0.62rem; font-weight:700; opacity:0.7; text-transform:uppercase; letter-spacing:0.06em; }
    .bio-pill .bvalue  { font-size:1.15rem; font-weight:700; margin:0.2rem 0; }
    .bio-pill .bstatus { font-size:0.68rem; font-weight:600; }
    .bio-normal { background:#f0fdf4; color:#166534; border:1px solid #86efac; }
    .bio-pre    { background:#fffbeb; color:#92400e; border:1px solid #fde68a; }
    .bio-danger { background:#fef2f2; color:#991b1b; border:1px solid #fca5a5; }

    /* ── Section & analysis titles ── */
    .section-title {
        font-size:1.05rem; font-weight:700; color:#0f2942;
        margin:1.5rem 0 0.75rem; display:flex; align-items:center; gap:0.5rem;
    }
    .analysis-title {
        font-size:0.82rem; font-weight:700; color:#0f2942;
        margin:1rem 0 0.3rem; display:flex; align-items:center; gap:0.4rem;
        text-transform:uppercase; letter-spacing:0.05em;
    }

    /* ── Inputs ── */
    .stTextInput input, .stDateInput input, .stNumberInput input {
        border-radius:8px !important; border:1.5px solid #e2e8f0 !important;
        background:#f8fafc !important; color:#0f2942 !important; font-weight:500 !important;
    }
    .stTextInput input:focus, .stDateInput input:focus, .stNumberInput input:focus {
        border-color:#0d6e6e !important; box-shadow:0 0 0 3px rgba(13,110,110,0.1) !important;
    }

    /* ── Buttons ── */
    .stButton > button, .stFormSubmitButton > button {
        background:linear-gradient(135deg, #0f2942, #0d6e6e) !important;
        color:white !important; border:none !important; border-radius:10px !important;
        font-weight:600 !important; padding:0.6rem 1.5rem !important;
        transition:opacity 0.2s !important; font-size:0.9rem !important;
    }
    .stButton > button:hover, .stFormSubmitButton > button:hover { opacity:0.88 !important; }

    [data-testid="stDataFrame"] { border-radius:12px; overflow:hidden; border:1px solid #e2e8f0; }
    .stAlert { border-radius:10px !important; }

    /* ── Patient info grid ── */
    .info-grid { display:grid; grid-template-columns:1fr 1fr; gap:0.75rem; }
    .info-item { background:#f8fafc; border-radius:8px; padding:0.65rem 0.9rem; border:1px solid #e2e8f0; }
    .info-item .info-label { font-size:0.62rem; font-weight:700; text-transform:uppercase; letter-spacing:0.07em; color:#64748b; }
    .info-item .info-value { font-size:0.92rem; font-weight:600; color:#0f2942; margin-top:0.15rem; }

    h3 { color:#0f2942 !important; font-weight:700 !important; }
    h4 { color:#1e3a5f !important; font-weight:600 !important; }
    #MainMenu, footer { visibility:hidden; }
</style>
""", unsafe_allow_html=True)

# ── Header ────────────────────────────────────────────────────────────────────
st.markdown("""
<div class="main-header">
    <h1>🏥 Health Prediction System</h1>
    <p>AI/ML-Powered Diabetes Risk Assessment · Patient Blood Test Analysis</p>
    <div class="header-badges">
        <span class="badge">🤖 Random Forest · 100 Trees</span>
        <span class="badge">📊 Kaggle Diabetes Dataset 2024</span>
        <span class="badge">⚕️ Glucose · HbA1c · Cholesterol</span>
    </div>
</div>
""", unsafe_allow_html=True)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("## 📋 Navigation")
    page = st.radio("", ["📊 Dashboard", "➕ Add Patient", "✏️ Edit / Update", "🗑️ Delete Patient"],
                    label_visibility="collapsed")
    st.markdown("---")
    st.markdown("**⚙️ Tech Stack**")
    st.markdown("- Python 3.x\n- Streamlit\n- SQLAlchemy + SQLite\n- scikit-learn")
    st.markdown("---")
    st.markdown("**📁 Dataset**")
    st.markdown("Kaggle · rabieelkharoua\nDiabetes Health Dataset 2024")
    st.markdown("---")
    st.caption("Health Prediction System v2.0")


# ── Helpers ───────────────────────────────────────────────────────────────────
def calc_age(dob):
    today = date.today()
    return today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

def run_prediction(age, glucose, haemoglobin, cholesterol):
    return predict_health(float(age), float(glucose), float(haemoglobin), float(cholesterol))

def glucose_status(g):
    if g < 70:     return ("Low",          "bio-danger")
    elif g <= 99:  return ("Normal",        "bio-normal")
    elif g <= 125: return ("Pre-Diabetic",  "bio-pre")
    else:          return ("Diabetic",      "bio-danger")

def hba1c_status(h):
    if h < 5.7:    return ("Normal",        "bio-normal")
    elif h <= 6.4: return ("Pre-Diabetic",  "bio-pre")
    else:          return ("Diabetic",      "bio-danger")

def chol_status(c):
    if c < 200:    return ("Desirable",     "bio-normal")
    elif c <= 239: return ("Borderline",    "bio-pre")
    else:          return ("High Risk",     "bio-danger")

def risk_class(remarks):
    r = remarks.upper()
    if "HIGH RISK" in r and "MODERATE" not in r: return "risk-high"
    if "MODERATE" in r or "MILD" in r:           return "risk-mild"
    return "risk-low"

def render_biomarkers(glucose, haemoglobin, cholesterol):
    gs, gc = glucose_status(glucose)
    hs, hc = hba1c_status(haemoglobin)
    cs, cc = chol_status(cholesterol)
    st.markdown(f"""
    <div class="biomarker-row">
        <div class="bio-pill {gc}">
            <span class="blabel">Glucose</span>
            <span class="bvalue">{glucose} mg/dL</span>
            <span class="bstatus">● {gs}</span>
        </div>
        <div class="bio-pill {hc}">
            <span class="blabel">HbA1c</span>
            <span class="bvalue">{haemoglobin}%</span>
            <span class="bstatus">● {hs}</span>
        </div>
        <div class="bio-pill {cc}">
            <span class="blabel">Cholesterol</span>
            <span class="bvalue">{cholesterol} mg/dL</span>
            <span class="bstatus">● {cs}</span>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_patient_info(p):
    st.markdown(f"""
    <div class="info-grid">
        <div class="info-item">
            <div class="info-label">Full Name</div>
            <div class="info-value">{p.full_name}</div>
        </div>
        <div class="info-item">
            <div class="info-label">Age</div>
            <div class="info-value">{p.age} yrs</div>
        </div>
        <div class="info-item">
            <div class="info-label">Date of Birth</div>
            <div class="info-value">{p.date_of_birth}</div>
        </div>
        <div class="info-item">
            <div class="info-label">Email</div>
            <div class="info-value">{p.email}</div>
        </div>
    </div>
    """, unsafe_allow_html=True)

def render_analysis(remarks, glucose, haemoglobin, cholesterol):
    rc = risk_class(remarks)
    st.markdown('<div class="analysis-title">🩸 Biomarker Summary</div>', unsafe_allow_html=True)
    render_biomarkers(glucose, haemoglobin, cholesterol)
    st.markdown('<div class="analysis-title">📋 Clinical Assessment Report</div>', unsafe_allow_html=True)
    st.markdown(f'<div class="risk-box {rc}">{remarks}</div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════════════════
# PAGE: DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
if page == "📊 Dashboard":
    db = get_session()
    patients = get_all_patients(db)
    db.close()

    if not patients:
        st.info("No patient records yet. Use ➕ Add Patient to get started.")
    else:
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Patients",  len(patients))
        c2.metric("Avg Glucose",     f"{sum(p.glucose for p in patients)/len(patients):.1f} mg/dL")
        c3.metric("Avg HbA1c",       f"{sum(p.haemoglobin for p in patients)/len(patients):.1f} %")
        c4.metric("Avg Cholesterol", f"{sum(p.cholesterol for p in patients)/len(patients):.1f} mg/dL")

        st.markdown('<div class="section-title">📋 All Patient Records</div>', unsafe_allow_html=True)
        df = pd.DataFrame([{
            "ID": p.id, "Full Name": p.full_name, "Age": p.age,
            "DOB": str(p.date_of_birth), "Email": p.email,
            "Glucose (mg/dL)": p.glucose,
            "HbA1c (%)": p.haemoglobin,
            "Cholesterol (mg/dL)": p.cholesterol,
        } for p in patients])
        st.dataframe(df, use_container_width=True, hide_index=True)

        st.markdown('<div class="section-title">🔍 Patient Detail & Analysis</div>', unsafe_allow_html=True)
        opts = {f"#{p.id} — {p.full_name}": p.id for p in patients}
        sel_id = opts[st.selectbox("Select a patient to view", list(opts.keys()))]
        db = get_session()
        p = get_patient_by_id(db, sel_id)
        db.close()
        if p:
            col1, col2 = st.columns([1, 1.6])
            with col1:
                st.markdown('<div class="card"><div class="card-title">👤 Patient Info</div>', unsafe_allow_html=True)
                render_patient_info(p)
                st.markdown('</div>', unsafe_allow_html=True)
            with col2:
                st.markdown('<div class="card"><div class="card-title">🩸 Biomarkers & Clinical Report</div>', unsafe_allow_html=True)
                render_analysis(p.remarks or "No analysis available.", p.glucose, p.haemoglobin, p.cholesterol)
                st.markdown('</div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════════════════
# PAGE: ADD PATIENT
# ══════════════════════════════════════════════════════════════════════════════
elif page == "➕ Add Patient":
    st.markdown('<div class="section-title">➕ Register New Patient</div>', unsafe_allow_html=True)

    with st.form("add_form", clear_on_submit=True):
        st.markdown("#### 👤 Personal Information")
        c1, c2 = st.columns(2)
        full_name = c1.text_input("Full Name *", placeholder="e.g. John Smith")
        email     = c2.text_input("Email Address *", placeholder="e.g. patient@email.com")
        dob       = st.date_input("Date of Birth *", value=date(1990, 1, 1),
                                   min_value=date(1900, 1, 1), max_value=date.today())
        st.markdown("#### 🩸 Blood Test Results")
        st.caption("Reference ranges based on WHO / ADA clinical guidelines")
        b1, b2, b3 = st.columns(3)
        glucose     = b1.number_input("Glucose (mg/dL) *", min_value=50.0, max_value=500.0,
                                       value=90.0, step=0.1, format="%.1f",
                                       help="Fasting glucose · Normal: 70–99 mg/dL")
        haemoglobin = b2.number_input("HbA1c (%) *", min_value=3.0, max_value=15.0,
                                       value=5.5, step=0.1, format="%.1f",
                                       help="Normal <5.7% | Pre-DM 5.7–6.4% | DM ≥6.5%")
        cholesterol = b3.number_input("Cholesterol (mg/dL) *", min_value=100.0, max_value=400.0,
                                       value=185.0, step=0.1, format="%.1f",
                                       help="Desirable <200 | Borderline 200–239 | High ≥240")
        submitted = st.form_submit_button("🔬 Analyse & Save Patient", use_container_width=True)

    if submitted:
        is_valid, errors = validate_patient_inputs(full_name, dob, email, glucose, haemoglobin, cholesterol)
        if not is_valid:
            for e in errors: st.error(e)
        else:
            db = get_session()
            existing = get_patient_by_email(db, email)
            if existing:
                st.error(f"A patient with email {email} already exists (ID #{existing.id}).")
                db.close()
            else:
                age = calc_age(dob)
                with st.spinner("🤖 Running clinical ML analysis..."):
                    remarks = run_prediction(age, glucose, haemoglobin, cholesterol)
                patient = create_patient(db, full_name, dob, email, age,
                                          float(glucose), float(haemoglobin), float(cholesterol), remarks)
                db.close()
                st.success(f"✅ {patient.full_name} registered successfully — ID #{patient.id} · Age {age}")
                st.markdown('<div class="card">', unsafe_allow_html=True)
                render_analysis(remarks, glucose, haemoglobin, cholesterol)
                st.markdown('</div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════════════════
# PAGE: EDIT / UPDATE
# ══════════════════════════════════════════════════════════════════════════════
elif page == "✏️ Edit / Update":
    st.markdown('<div class="section-title">✏️ Edit Patient Record</div>', unsafe_allow_html=True)
    db = get_session()
    patients = get_all_patients(db)
    db.close()

    if not patients:
        st.info("No records to edit.")
    else:
        opts = {f"#{p.id} — {p.full_name} ({p.email})": p.id for p in patients}
        sel_id = opts[st.selectbox("Select patient to edit", list(opts.keys()))]
        db = get_session()
        p = get_patient_by_id(db, sel_id)
        db.close()

        if p:
            with st.form("edit_form"):
                st.markdown("#### 👤 Personal Information")
                c1, c2 = st.columns(2)
                full_name = c1.text_input("Full Name *", value=p.full_name)
                email     = c2.text_input("Email *", value=p.email)
                dob       = st.date_input("Date of Birth *", value=p.date_of_birth,
                                           min_value=date(1900, 1, 1), max_value=date.today())
                st.markdown("#### 🩸 Blood Test Results")
                b1, b2, b3 = st.columns(3)
                glucose     = b1.number_input("Glucose (mg/dL) *", value=float(p.glucose),
                                               min_value=50.0, max_value=500.0, step=0.1, format="%.1f")
                haemoglobin = b2.number_input("HbA1c (%) *", value=float(p.haemoglobin),
                                               min_value=3.0, max_value=15.0, step=0.1, format="%.1f")
                cholesterol = b3.number_input("Cholesterol (mg/dL) *", value=float(p.cholesterol),
                                               min_value=100.0, max_value=400.0, step=0.1, format="%.1f")
                update_btn = st.form_submit_button("💾 Update & Re-Analyse", use_container_width=True)

            if update_btn:
                is_valid, errors = validate_patient_inputs(full_name, dob, email, glucose, haemoglobin, cholesterol)
                if not is_valid:
                    for e in errors: st.error(e)
                else:
                    db = get_session()
                    ex = get_patient_by_email(db, email)
                    if ex and ex.id != sel_id:
                        st.error(f"Email {email} already belongs to another patient.")
                        db.close()
                    else:
                        age = calc_age(dob)
                        with st.spinner("🤖 Running clinical ML analysis..."):
                            remarks = run_prediction(age, glucose, haemoglobin, cholesterol)
                        update_patient(db, sel_id, full_name=full_name, date_of_birth=dob,
                                        email=email, age=age, glucose=float(glucose),
                                        haemoglobin=float(haemoglobin),
                                        cholesterol=float(cholesterol), remarks=remarks)
                        db.close()
                        st.success(f"✅ Patient #{sel_id} updated successfully!")
                        st.markdown('<div class="card">', unsafe_allow_html=True)
                        render_analysis(remarks, glucose, haemoglobin, cholesterol)
                        st.markdown('</div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════════════════
# PAGE: DELETE
# ══════════════════════════════════════════════════════════════════════════════
elif page == "🗑️ Delete Patient":
    st.markdown('<div class="section-title">🗑️ Delete Patient Record</div>', unsafe_allow_html=True)
    db = get_session()
    patients = get_all_patients(db)
    db.close()

    if not patients:
        st.info("No records to delete.")
    else:
        opts = {f"#{p.id} — {p.full_name} ({p.email})": p.id for p in patients}
        sel_id = opts[st.selectbox("Select patient to delete", list(opts.keys()))]
        db = get_session()
        p = get_patient_by_id(db, sel_id)
        db.close()

        if p:
            st.markdown('<div class="card"><div class="card-title">👤 Patient to be Deleted</div>', unsafe_allow_html=True)
            render_patient_info(p)
            st.markdown('</div>', unsafe_allow_html=True)
            st.warning(f"⚠️ You are about to permanently delete **{p.full_name}** (ID #{p.id}). This cannot be undone.")
            confirm = st.checkbox("I understand and confirm this deletion")
            if st.button("🗑️ Delete Patient", disabled=not confirm):
                db = get_session()
                delete_patient(db, sel_id)
                db.close()
                st.success(f"✅ {p.full_name}'s record has been deleted.")
                st.rerun()

Overwriting app.py


In [26]:
!python train_model.py

Generating dataset based on WHO/ADA/Kaggle clinical distributions...
Accuracy: 100.00%
              precision    recall  f1-score   support

 No Diabetes       1.00      1.00      1.00       750
    Diabetic       1.00      1.00      1.00       250

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

Model saved!


In [27]:
import subprocess, time
subprocess.run(["pkill", "-f", "streamlit"],   capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(3)
print("✅ Cleared")

✅ Cleared


In [28]:
import subprocess, time, urllib.request
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false"
])
time.sleep(6)
try:
    urllib.request.urlopen("http://localhost:8501")
    print("✅ Streamlit is running!")
except:
    print("❌ Streamlit failed")

✅ Streamlit is running!


In [29]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("✅ cloudflared ready")

✅ cloudflared ready


In [30]:
import subprocess, time, re
proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print("⏳ Waiting for URL...")
for _ in range(60):
    line = proc.stdout.readline().decode("utf-8", errors="ignore")
    if "trycloudflare.com" in line:
        url = re.search(r"https://[^\s]+trycloudflare\.com", line)
        if url:
            print(f"\n✅ YOUR APP IS LIVE AT:")
            print(f"👉  {url.group(0)}")
            print(f"\nKeep this cell running!")
            break
    time.sleep(1)

⏳ Waiting for URL...

✅ YOUR APP IS LIVE AT:
👉  https://podcasts-fought-assumption-specializing.trycloudflare.com

Keep this cell running!
